# Notebook 04: Modelado de LSTM
*Rodrigo Ajmac Aroche - 22279*

*June Herrera - 21749* 

*Andres Mazariegos - 231038*

## 0. CONFIGURACIÓN

**Librerías**

In [9]:
import numpy as np
import pandas as pd
import itertools
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error
import warnings
warnings.filterwarnings('ignore')

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader

**Funciones**

In [10]:
def serie_de(mascara):
    return (base[mascara].groupby('fecha')['Viajero'].sum()
            .reindex(IDX).fillna(0).astype(float))

def create_sequences(data, lookback):
    X, y = [], []
    for i in range(len(data) - lookback):
        X.append(data[i:(i + lookback), 0])
        y.append(data[i + lookback, 0])
    return np.array(X), np.array(y)

class TimeSeriesLSTM(nn.Module):
    def __init__(self, input_size=1, hidden_layer_size=50, num_layers=1, dropout_rate=0.2):
        super().__init__()
        dropout = dropout_rate if num_layers > 1 else 0.0
        self.lstm = nn.LSTM(
            input_size=input_size, 
            hidden_size=hidden_layer_size, 
            num_layers=num_layers, 
            batch_first=True, 
            dropout=dropout
        )
        self.linear = nn.Linear(hidden_layer_size, 1)

    def forward(self, input_seq):
        lstm_out, _ = self.lstm(input_seq)
        predictions = self.linear(lstm_out[:, -1, :])
        return predictions

def optimizar_serie_pytorch(nombre, serie, corte, lookback, grid):
    print(f"{'='*60}\nOptimizando modelos (PyTorch) para: {nombre}\n{'='*60}")
    
    train_data = serie.iloc[:corte].values.reshape(-1, 1)
    test_data = serie.iloc[corte:].values.reshape(-1, 1)
    
    scaler = MinMaxScaler(feature_range=(0, 1))
    train_scaled = scaler.fit_transform(train_data)
    test_scaled = scaler.transform(test_data)
    
    X_train, y_train = create_sequences(train_scaled, lookback)
    X_train = np.reshape(X_train, (X_train.shape[0], X_train.shape[1], 1))
    
    combined_test = np.vstack((train_scaled[-lookback:], test_scaled))
    X_test, y_test = create_sequences(combined_test, lookback)
    X_test = np.reshape(X_test, (X_test.shape[0], X_test.shape[1], 1))
    
    X_train_tensor = torch.tensor(X_train, dtype=torch.float32)
    y_train_tensor = torch.tensor(y_train, dtype=torch.float32).view(-1, 1)
    X_test_tensor = torch.tensor(X_test, dtype=torch.float32)
    
    mejores = []
    
    for i, p in enumerate(grid):
        model = TimeSeriesLSTM(
            input_size=1, 
            hidden_layer_size=p['units'], 
            num_layers=p['lstm_layers']
        )
        loss_function = nn.MSELoss()
        optimizer = optim.Adam(model.parameters(), lr=0.001)
        
        dataset = TensorDataset(X_train_tensor, y_train_tensor)
        loader = DataLoader(dataset, batch_size=p['batch_size'], shuffle=False)
        
        model.train()
        for epoch in range(p['epochs']):
            for batch_X, batch_y in loader:
                optimizer.zero_grad()
                y_pred = model(batch_X)
                loss = loss_function(y_pred, batch_y)
                loss.backward()
                optimizer.step()
        
        model.eval()
        with torch.no_grad():
            preds_scaled = model(X_test_tensor).numpy()
            
        preds = scaler.inverse_transform(preds_scaled).flatten()
        preds = np.clip(preds, 0, None)
        real = serie.iloc[corte:].values
        
        rmse = np.sqrt(mean_squared_error(real, preds))
        mae = mean_absolute_error(real, preds)
        mejores.append({'Config': p, 'RMSE': rmse, 'MAE': mae, 'Modelo': model, 'Preds': preds})
    
    top_2 = sorted(mejores, key=lambda x: x['RMSE'])[:2]
    print(f"  TOP 1 -> RMSE: {top_2[0]['RMSE']:,.0f} | MAE: {top_2[0]['MAE']:,.0f}")
    print(f"           (Capas: {top_2[0]['Config']['lstm_layers']}, Unidades: {top_2[0]['Config']['units']}, Épocas: {top_2[0]['Config']['epochs']}, Lote: {top_2[0]['Config']['batch_size']})")
    
    print(f"  TOP 2 -> RMSE: {top_2[1]['RMSE']:,.0f} | MAE: {top_2[1]['MAE']:,.0f}")
    print(f"           (Capas: {top_2[1]['Config']['lstm_layers']}, Unidades: {top_2[1]['Config']['units']}, Épocas: {top_2[1]['Config']['epochs']}, Lote: {top_2[1]['Config']['batch_size']})\n")
    
    return top_2

In [11]:
RUTA = '../Datos_Crudos/Base_Migracion_2009-2026jun.xlsx'

## 1. PREPROCESAMIENTO

In [12]:
# ==========================================
# 1. CARGA Y CONSTRUCCIÓN DE LAS SERIES
# ==========================================
df = pd.read_excel(RUTA, sheet_name=0)
df['Viajero'] = df['Viajero'].astype(int)
df['fecha'] = pd.to_datetime(
    df['Año'].astype(str) + '-' + df['Mes cod'].astype(str).str.zfill(2) + '-01')

base = df
IDX = pd.date_range('2009-01-01', '2026-06-01', freq='MS')

In [13]:
rank_pais = base.groupby('País')['Viajero'].sum().sort_values(ascending=False)
top3_pais = list(rank_pais.head(3).index)
VIAS = ['Aérea', 'Terrestre', 'Marítima']

TODO = pd.Series(True, index=base.index)
MASCARAS = {'Total de viajeros': TODO}
for p in top3_pais:
    MASCARAS[f'País: {p}'] = base['País'] == p
for v in VIAS:
    MASCARAS[f'Vía: {v}'] = base['Vía'] == v

SERIES = {n: serie_de(m) for n, m in MASCARAS.items()}
print(f"Se construyeron {len(SERIES)} series correctamente.\n")

Se construyeron 7 series correctamente.



## 2. FINETUNING

In [14]:
N = len(IDX)
CORTE = int(N * 0.70)
LOOKBACK = 12
resultados_lstm = {}

param_grid = {
    'lstm_layers': [1, 2],
    'units': [32, 64],
    'epochs': [50, 100],
    'batch_size': [8, 16]
}
keys, values = zip(*param_grid.items())
grid_combinaciones = [dict(zip(keys, v)) for v in itertools.product(*values)]

In [15]:
# ==========================================
# 4. EJECUCIÓN DEL MODELADO SOBRE TODAS LAS SERIES
# ==========================================
for nombre_serie, serie_datos in SERIES.items():
    resultados_lstm[nombre_serie] = optimizar_serie_pytorch(
        nombre=nombre_serie, 
        serie=serie_datos, 
        corte=CORTE, 
        lookback=LOOKBACK, 
        grid=grid_combinaciones
    )

print("Proceso de modelado finalizado exitosamente.")

Optimizando modelos (PyTorch) para: Total de viajeros
  TOP 1 -> RMSE: 54,408 | MAE: 40,935
           (Capas: 1, Unidades: 64, Épocas: 100, Lote: 8)
  TOP 2 -> RMSE: 56,244 | MAE: 43,400
           (Capas: 1, Unidades: 32, Épocas: 100, Lote: 8)

Optimizando modelos (PyTorch) para: País: El Salvador
  TOP 1 -> RMSE: 26,865 | MAE: 20,671
           (Capas: 1, Unidades: 64, Épocas: 100, Lote: 8)
  TOP 2 -> RMSE: 26,906 | MAE: 19,678
           (Capas: 1, Unidades: 32, Épocas: 100, Lote: 8)

Optimizando modelos (PyTorch) para: País: Guatemala
  TOP 1 -> RMSE: 27,828 | MAE: 14,126
           (Capas: 1, Unidades: 64, Épocas: 100, Lote: 16)
  TOP 2 -> RMSE: 28,694 | MAE: 16,170
           (Capas: 1, Unidades: 32, Épocas: 100, Lote: 8)

Optimizando modelos (PyTorch) para: País: Estados Unidos de América
  TOP 1 -> RMSE: 9,394 | MAE: 7,701
           (Capas: 2, Unidades: 64, Épocas: 100, Lote: 8)
  TOP 2 -> RMSE: 9,697 | MAE: 7,467
           (Capas: 1, Unidades: 64, Épocas: 100, Lote: 8)

Opt